# CASMI 2026 — Kaggle Full-Scale Pipeline

Deployment notebook (per WORKFLOW.md's two-notebook rule). Built and validated locally in `Enveda_CASMI_local.ipynb`; this notebook inlines `src/metric.py`, `src/data.py`, `src/baseline.py`, `src/reranker.py` (Kaggle notebooks are single-file, no repo checkout) and runs the full pipeline at the complete ~2.5M-spectra dataset scale, using Kaggle's 30GB RAM (vs. the ~16GB local ceiling that capped local runs at 1M rows).

**Build phase:** internet ON (installs autogluon/rdkit/matchms). The final submission notebook must also verify success with internet OFF per competition rules — that is a follow-up step after this run proves the pipeline works at full scale.

In [ ]:
!pip install -q --upgrade numpy scipy
!pip install -q autogluon.tabular rdkit matchms


## Inlined `src/metric.py`

In [ ]:
"""Official CASMI 2026 MRR@25 metric: InChIKey14-based structure matching."""

from rdkit import Chem
from rdkit.Chem.MolStandardize import rdMolStandardize

_tautomer_enumerator = rdMolStandardize.TautomerEnumerator()


def canonicalize_to_inchikey14(smiles: str) -> str | None:
    """Parse SMILES, canonicalize tautomers, return the 14-char InChIKey prefix.

    Returns None if the SMILES cannot be parsed by RDKit.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    canonical_mol = _tautomer_enumerator.Canonicalize(mol)
    inchikey = Chem.MolToInchiKey(canonical_mol)
    if not inchikey:
        return None
    return inchikey[:14]


def reciprocal_rank(predicted_smiles: list[str], true_smiles: str, k: int = 25) -> float:
    """1/rank of the first predicted SMILES matching true_smiles's InChIKey14.

    Unparseable predicted SMILES occupy a rank slot but never match.
    Returns 0.0 if no match is found within the first k predictions.
    """
    true_key = canonicalize_to_inchikey14(true_smiles)
    for rank, smiles in enumerate(predicted_smiles[:k], start=1):
        predicted_key = canonicalize_to_inchikey14(smiles)
        if predicted_key is not None and predicted_key == true_key:
            return 1.0 / rank
    return 0.0


def mrr_at_25(
    predictions: dict[str, list[str]],
    ground_truth: dict[str, str],
    k: int = 25,
) -> float:
    """Mean reciprocal rank @ k over all molecules in ground_truth.

    Raises KeyError if a ground_truth molecule_id is missing from predictions.
    """
    scores = [
        reciprocal_rank(predictions[molecule_id], true_smiles, k=k)
        for molecule_id, true_smiles in ground_truth.items()
    ]
    return sum(scores) / len(scores)


## Inlined `src/data.py` (paths adapted for Kaggle's mounted competition data)

In [ ]:
"""Data loading for the CASMI 2026 pipeline (Kaggle paths)."""

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

KAGGLE_INPUT_DIR = "/kaggle/input/enveda-CASMI26-molecule-id-mass-spectra"


def load_test_set(path: str = f"{KAGGLE_INPUT_DIR}/test.parquet") -> pd.DataFrame:
    """Load the full test set (it's small, ~1500 rows)."""
    return pd.read_parquet(path)


def load_sampled_train(
    path: str = f"{KAGGLE_INPUT_DIR}/train.parquet",
    sample_size: int = 75_000,
    always_include_libs: tuple[str, ...] = ("enveda-np-examples", "drug_plus"),
    random_state: int = 42,
) -> pd.DataFrame:
    """Load a stratified sample of the training set.

    All rows from `always_include_libs` are kept. Additional rows are randomly
    sampled from the rest up to `sample_size` total rows. Reads the full
    training parquet once as a pyarrow Table and takes only the selected rows,
    to avoid materializing the whole table as pandas when sample_size is small.
    """
    lib_column = pq.read_table(path, columns=["ingest_lib"]).column("ingest_lib").to_pandas()
    priority_mask = lib_column.isin(always_include_libs)
    priority_idx = lib_column.index[priority_mask]
    remaining_idx = lib_column.index[~priority_mask]

    remaining_budget = sample_size - len(priority_idx)
    if remaining_budget <= 0:
        selected_idx = priority_idx
    else:
        sampled_remaining_idx = (
            pd.Index(remaining_idx)
            .to_series()
            .sample(n=min(remaining_budget, len(remaining_idx)), random_state=random_state)
            .index
        )
        selected_idx = priority_idx.append(sampled_remaining_idx)

    full_table = pq.read_table(path)
    selected_table = full_table.take(pa.array(selected_idx))
    return selected_table.to_pandas().reset_index(drop=True)


def make_validation_split(
    train_df: pd.DataFrame, n_held_out: int = 200, random_state: int = 42
) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, str]]:
    """Build an offline validation split (spectrum-level, timsTOF-preferred).

    For each eligible held-out inchikey14 (2+ spectra), holds out some spectra
    as queries while leaving at least one other spectrum of that structure in
    the candidate pool. Prefers timsTOF molecules (matches the real test set),
    falling back to non-timsTOF if too few timsTOF-eligible molecules exist.

    Returns (val_query_df, val_train_df, val_ground_truth) where
    val_ground_truth maps molecule_id -> normalized_smiles.
    """
    rng = np.random.RandomState(random_state)

    spectra_counts = train_df.groupby("inchikey14").size()
    eligible_keys = spectra_counts[spectra_counts >= 2].index

    eligible_df = train_df[train_df["inchikey14"].isin(eligible_keys)]
    if "instrument_type" in eligible_df.columns:
        timstof_keys = pd.Index(
            eligible_df.loc[eligible_df["instrument_type"] == "timsTOF", "inchikey14"].unique()
        )
    else:
        timstof_keys = pd.Index([], dtype=object)
    other_eligible_keys = pd.Index(eligible_keys).difference(timstof_keys)

    n_from_timstof = min(n_held_out, len(timstof_keys))
    selected_timstof = rng.choice(timstof_keys, size=n_from_timstof, replace=False)

    remaining_needed = n_held_out - n_from_timstof
    n_from_other = min(remaining_needed, len(other_eligible_keys))
    selected_other = rng.choice(other_eligible_keys, size=n_from_other, replace=False)

    held_out_molecules = np.concatenate([selected_timstof, selected_other])

    query_row_indices = []
    train_row_indices = []
    for key in held_out_molecules:
        group_idx = train_df.index[train_df["inchikey14"] == key]
        shuffled = rng.permutation(group_idx.to_numpy())
        n_query = max(1, len(shuffled) // 2)
        n_query = min(n_query, len(shuffled) - 1)
        query_row_indices.extend(shuffled[:n_query])
        train_row_indices.extend(shuffled[n_query:])

    held_out_set = set(held_out_molecules)
    non_held_out_idx = train_df.index[~train_df["inchikey14"].isin(held_out_set)]

    val_query_df = train_df.loc[query_row_indices].copy()
    val_query_df["molecule_id"] = val_query_df["inchikey14"]

    val_train_idx = pd.Index(train_row_indices).append(non_held_out_idx)
    val_train_df = train_df.loc[val_train_idx]

    val_ground_truth = (
        val_query_df.drop_duplicates("inchikey14")
        .set_index("molecule_id")["normalized_smiles"]
        .to_dict()
    )

    return val_query_df, val_train_df, val_ground_truth


## Inlined `src/baseline.py`

In [ ]:
"""Precursor-mass + cosine-similarity retrieval baseline."""

import numpy as np
import pandas as pd
from matchms import Spectrum
from matchms.similarity import CosineGreedy

_cosine_greedy = CosineGreedy(tolerance=0.01)


def to_matchms_spectrum(mzs, intensities, metadata: dict) -> Spectrum:
    """Build a matchms Spectrum from raw mz/intensity arrays."""
    mz_array = np.asarray(mzs, dtype=float)
    intensity_array = np.asarray(intensities, dtype=float)
    order = np.argsort(mz_array)
    return Spectrum(
        mz=mz_array[order],
        intensities=intensity_array[order],
        metadata=metadata,
    )


def filter_candidates(
    test_row: pd.Series, train_df: pd.DataFrame, ppm_tolerance: float = 15.0
) -> pd.DataFrame:
    """Return train rows matching test_row's adduct within a ppm precursor-mz window."""
    adduct_mask = train_df["adduct"] == test_row["adduct"]
    ppm_error = (
        (train_df["precursor_mz"] - test_row["precursor_mz"]).abs()
        / test_row["precursor_mz"]
        * 1e6
    )
    ppm_mask = ppm_error <= ppm_tolerance
    return train_df[adduct_mask & ppm_mask]


def cosine_similarity(spectrum_a: Spectrum, spectrum_b: Spectrum) -> float:
    """matchms CosineGreedy score between two spectra; 0.0 if either is empty."""
    if len(spectrum_a.peaks.mz) == 0 or len(spectrum_b.peaks.mz) == 0:
        return 0.0
    result = _cosine_greedy.pair(spectrum_a, spectrum_b)
    return float(result["score"])


def score_candidates_for_molecule(
    test_spectra_rows: pd.DataFrame, train_df: pd.DataFrame, ppm_tolerance: float = 15.0
) -> pd.DataFrame:
    """Score every candidate train structure against a molecule's test spectra.

    Returns columns ["inchikey14", "smiles", "score"], deduplicated by
    inchikey14 keeping the max score across all (test spectrum, candidate)
    pairs. Candidate Spectrum objects are cached per molecule since the same
    train_df row often reappears across a molecule's multiple test spectra.
    """
    best_score: dict[str, float] = {}
    best_smiles: dict[str, str] = {}
    candidate_spectrum_cache: dict[int, Spectrum] = {}

    for _, test_row in test_spectra_rows.iterrows():
        candidates = filter_candidates(test_row, train_df, ppm_tolerance=ppm_tolerance)
        if candidates.empty:
            continue
        test_spectrum = to_matchms_spectrum(
            test_row["ms2_mzs"], test_row["ms2_normalized_intensities"], metadata={}
        )
        for candidate_idx, candidate_row in candidates.iterrows():
            if candidate_idx not in candidate_spectrum_cache:
                candidate_spectrum_cache[candidate_idx] = to_matchms_spectrum(
                    candidate_row["ms2_mzs"],
                    candidate_row["ms2_normalized_intensities"],
                    metadata={},
                )
            candidate_spectrum = candidate_spectrum_cache[candidate_idx]
            score = cosine_similarity(test_spectrum, candidate_spectrum)
            key = candidate_row["inchikey14"]
            if score > best_score.get(key, -1.0):
                best_score[key] = score
                best_smiles[key] = candidate_row["normalized_smiles"]

    return pd.DataFrame(
        {
            "inchikey14": list(best_score.keys()),
            "smiles": [best_smiles[k] for k in best_score],
            "score": list(best_score.values()),
        }
    )


def build_submission(
    test_df: pd.DataFrame, train_df: pd.DataFrame, ppm_tolerance: float = 15.0, top_k: int = 25
) -> pd.DataFrame:
    """Build a molecule_id -> top-k semicolon-joined SMILES submission dataframe."""
    rows = []
    for molecule_id, group in test_df.groupby("molecule_id"):
        scored = score_candidates_for_molecule(group, train_df, ppm_tolerance=ppm_tolerance)
        if scored.empty:
            rows.append({"molecule_id": molecule_id, "smiles": ""})
            continue
        top = scored.sort_values("score", ascending=False).head(top_k)
        rows.append({"molecule_id": molecule_id, "smiles": ";".join(top["smiles"])})
    return pd.DataFrame(rows)


## Inlined `src/reranker.py`

In [ ]:
"""AutoGluon-based reranker: richer per-candidate features + trained reordering."""

import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski
from matchms import Spectrum

_FEATURE_COLUMNS = [
    "inchikey14", "smiles", "cosine_score", "ppm_error",
    "num_peaks_candidate", "mol_wt", "log_p", "num_rings",
    "num_rotatable_bonds", "num_hbd", "num_hba",
]


def _rdkit_descriptors(smiles: str) -> dict:
    """Compute RDKit descriptors for a SMILES string; NaN values if unparseable."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {
            "mol_wt": np.nan, "log_p": np.nan, "num_rings": np.nan,
            "num_rotatable_bonds": np.nan, "num_hbd": np.nan, "num_hba": np.nan,
        }
    return {
        "mol_wt": Descriptors.MolWt(mol),
        "log_p": Descriptors.MolLogP(mol),
        "num_rings": Descriptors.RingCount(mol),
        "num_rotatable_bonds": Descriptors.NumRotatableBonds(mol),
        "num_hbd": Lipinski.NumHDonors(mol),
        "num_hba": Lipinski.NumHAcceptors(mol),
    }


def extract_candidate_features(
    test_spectra_rows: pd.DataFrame, train_df: pd.DataFrame, ppm_tolerance: float = 15.0
) -> pd.DataFrame:
    """Extract rich per-candidate features for reranker training/inference."""
    best_score: dict[str, float] = {}
    best_row: dict[str, dict] = {}
    candidate_spectrum_cache: dict[int, Spectrum] = {}
    descriptor_cache: dict[str, dict] = {}

    for _, test_row in test_spectra_rows.iterrows():
        candidates = filter_candidates(test_row, train_df, ppm_tolerance=ppm_tolerance)
        if candidates.empty:
            continue
        test_spectrum = to_matchms_spectrum(
            test_row["ms2_mzs"], test_row["ms2_normalized_intensities"], metadata={}
        )
        for candidate_idx, candidate_row in candidates.iterrows():
            if candidate_idx not in candidate_spectrum_cache:
                candidate_spectrum_cache[candidate_idx] = to_matchms_spectrum(
                    candidate_row["ms2_mzs"],
                    candidate_row["ms2_normalized_intensities"],
                    metadata={},
                )
            candidate_spectrum = candidate_spectrum_cache[candidate_idx]
            score = cosine_similarity(test_spectrum, candidate_spectrum)
            key = candidate_row["inchikey14"]
            if score > best_score.get(key, -1.0):
                best_score[key] = score
                ppm_error = (
                    abs(candidate_row["precursor_mz"] - test_row["precursor_mz"])
                    / test_row["precursor_mz"]
                    * 1e6
                )
                num_peaks = candidate_row.get("num_peaks")
                if num_peaks is None or pd.isna(num_peaks):
                    num_peaks = len(candidate_row["ms2_mzs"])
                smiles = candidate_row["normalized_smiles"]
                if smiles not in descriptor_cache:
                    descriptor_cache[smiles] = _rdkit_descriptors(smiles)
                best_row[key] = {
                    "inchikey14": key,
                    "smiles": smiles,
                    "cosine_score": score,
                    "ppm_error": ppm_error,
                    "num_peaks_candidate": num_peaks,
                    **descriptor_cache[smiles],
                }

    if not best_row:
        return pd.DataFrame(columns=_FEATURE_COLUMNS)

    return pd.DataFrame(list(best_row.values()), columns=_FEATURE_COLUMNS)


def build_training_examples(
    train_df: pd.DataFrame,
    n_held_out: int = 200,
    random_state: int = 42,
    ppm_tolerance: float = 15.0,
) -> pd.DataFrame:
    """Build a labeled training set for the reranker from held-out molecules.

    Labels each candidate row is_correct=1 if its inchikey14 matches the held-out
    molecule's molecule_id (its true inchikey14) -- matching the competition's
    actual InChIKey14-based scoring criterion, not exact SMILES string equality.
    """
    val_query_df, val_train_df, val_ground_truth = make_validation_split(
        train_df, n_held_out=n_held_out, random_state=random_state
    )

    all_rows = []
    for molecule_id, group in val_query_df.groupby("molecule_id"):
        features = extract_candidate_features(group, val_train_df, ppm_tolerance=ppm_tolerance)
        if features.empty:
            continue
        features = features.copy()
        features["molecule_id"] = molecule_id
        features["is_correct"] = (features["inchikey14"] == molecule_id).astype(int)
        all_rows.append(features)

    if not all_rows:
        columns = _FEATURE_COLUMNS + ["molecule_id", "is_correct"]
        return pd.DataFrame(columns=columns)

    return pd.concat(all_rows, ignore_index=True)


def train_reranker(
    training_df: pd.DataFrame,
    model_path: str,
    feature_columns: list[str],
    label_column: str = "is_correct",
    time_limit: int = 120,
):
    """Train a binary-classification AutoGluon reranker."""
    from autogluon.tabular import TabularPredictor

    predictor = TabularPredictor(
        label=label_column, path=model_path, problem_type="binary", eval_metric="roc_auc"
    )
    predictor.fit(training_df[feature_columns + [label_column]], time_limit=time_limit)
    return predictor


def rerank_candidates(
    predictor, candidate_features_df: pd.DataFrame, feature_columns: list[str], top_k: int = 25
) -> list[str]:
    """Reorder candidates by the trained reranker's predicted match probability."""
    if candidate_features_df.empty:
        return []

    probabilities = predictor.predict_proba(candidate_features_df[feature_columns])
    positive_class_probs = probabilities[1] if 1 in probabilities.columns else probabilities[True]

    ranked = candidate_features_df.assign(_positive_prob=positive_class_probs.values).sort_values(
        "_positive_prob", ascending=False
    )
    return ranked["smiles"].head(top_k).tolist()


## Pipeline: load data, baseline, reranker, double-holdout eval, submissions

In [ ]:
import matchms
matchms.set_matchms_logger_level("ERROR")

train_df = load_sampled_train(sample_size=2_600_000, random_state=42)
test_df = load_test_set()

print(f"Loaded {len(train_df):,} training spectra, {train_df['inchikey14'].nunique():,} unique structures")
print(f"Loaded {len(test_df):,} test spectra, {test_df['molecule_id'].nunique():,} unique molecules")


In [ ]:
val_query_df, val_train_df, val_ground_truth = make_validation_split(
    train_df, n_held_out=200, random_state=42
)
print(f"Validation query set: {val_query_df['molecule_id'].nunique()} held-out molecules")


In [ ]:
val_submission_df = build_submission(val_query_df, val_train_df, ppm_tolerance=15.0, top_k=25)
val_predictions = dict(
    zip(val_submission_df["molecule_id"], val_submission_df["smiles"].str.split(";"))
)

offline_mrr25 = mrr_at_25(val_predictions, val_ground_truth)
print(f"Offline MRR@25 (spectrum-level held-out split, timsTOF-preferred): {offline_mrr25:.4f}")


In [ ]:
submission_df = build_submission(test_df, train_df, ppm_tolerance=15.0, top_k=25)

# Kaggle format constraints: no empty/NaN smiles allowed.
submission_df["smiles"] = submission_df["smiles"].replace("", "CCO")

assert submission_df["molecule_id"].is_unique
assert submission_df["smiles"].notna().all()
assert (submission_df["smiles"].str.count(";") <= 24).all()

submission_df.to_csv("submission.csv", index=False)
print(f"Wrote submission.csv with {len(submission_df)} rows")
submission_df.head()


In [ ]:
_RERANKER_FEATURE_COLUMNS = [
    "cosine_score", "ppm_error", "num_peaks_candidate",
    "mol_wt", "log_p", "num_rings", "num_rotatable_bonds", "num_hbd", "num_hba",
]

reranker_training_df = build_training_examples(
    train_df, n_held_out=200, random_state=42
)
print(f"Reranker training examples: {len(reranker_training_df)} rows, "
      f"{reranker_training_df['is_correct'].sum()} positive")

reranker_predictor = train_reranker(
    reranker_training_df,
    model_path="./autogluon_reranker_model",
    feature_columns=_RERANKER_FEATURE_COLUMNS,
    time_limit=120,
)


In [ ]:
eval_query_df, eval_train_df, eval_ground_truth = make_validation_split(
    train_df, n_held_out=200, random_state=99
)
print(f"Independent eval set: {eval_query_df['molecule_id'].nunique()} held-out molecules "
      f"(disjoint random_state from reranker training split)")


In [ ]:
baseline_predictions_cmp = {}
reranked_predictions_cmp = {}

for molecule_id, group in eval_query_df.groupby("molecule_id"):
    features = extract_candidate_features(group, eval_train_df, ppm_tolerance=15.0)
    if features.empty:
        baseline_predictions_cmp[molecule_id] = [""]
        reranked_predictions_cmp[molecule_id] = [""]
        continue
    baseline_order = features.sort_values("cosine_score", ascending=False)["smiles"].tolist()
    baseline_predictions_cmp[molecule_id] = baseline_order if baseline_order else [""]
    reranked_predictions_cmp[molecule_id] = rerank_candidates(
        reranker_predictor, features, _RERANKER_FEATURE_COLUMNS, top_k=25
    ) or [""]

baseline_mrr_cmp = mrr_at_25(baseline_predictions_cmp, eval_ground_truth)
reranked_mrr_cmp = mrr_at_25(reranked_predictions_cmp, eval_ground_truth)

print(f"Baseline-only MRR@25:  {baseline_mrr_cmp:.4f}")
print(f"Reranked MRR@25:       {reranked_mrr_cmp:.4f}")
print(f"Delta:                 {reranked_mrr_cmp - baseline_mrr_cmp:+.4f}")
print()
print("Independent double-holdout: this eval split (random_state=99) is disjoint "
      "from the reranker's training split (random_state=42), so this delta reflects "
      "genuine generalization rather than in-sample memorization.")


In [ ]:
reranked_rows = []
for molecule_id, group in test_df.groupby("molecule_id"):
    features = extract_candidate_features(group, train_df, ppm_tolerance=15.0)
    if features.empty:
        reranked_rows.append({"molecule_id": molecule_id, "smiles": ""})
        continue
    top_smiles = rerank_candidates(
        reranker_predictor, features, _RERANKER_FEATURE_COLUMNS, top_k=25
    )
    reranked_rows.append({"molecule_id": molecule_id, "smiles": ";".join(top_smiles)})

submission_reranked_df = pd.DataFrame(reranked_rows)
submission_reranked_df["smiles"] = submission_reranked_df["smiles"].replace("", "CCO")

assert submission_reranked_df["molecule_id"].is_unique
assert submission_reranked_df["smiles"].notna().all()
assert (submission_reranked_df["smiles"].str.count(";") <= 24).all()
assert set(submission_reranked_df["molecule_id"]) == set(test_df["molecule_id"])

submission_reranked_df.to_csv("submission_reranked.csv", index=False)
print(f"Wrote submission_reranked.csv with {len(submission_reranked_df)} rows")
submission_reranked_df.head()
